# Topic Control with Llama 3.1 NemoGuard 8B TopicControl NIM

This notebook shows how to use the [Llama 3.1 NemoGuard 8B TopicControl NIM](https://docs.nvidia.com/nim/llama-3-1-nemoguard-8b-topiccontrol/latest/index.html) to restrict an LLM application to a defined set of allowed topics in NeMo Guardrails.

## Local Deployment

Pull and run both NIM containers. You need an NGC API key to pull the images —
obtain one at [ngc.nvidia.com](https://ngc.nvidia.com).

**Llama 3.1 NemoGuard 8B TopicControl NIM** (port 8123):

```bash
# Authenticate with NGC (username: $oauthtoken, password: your NGC API key)
docker login nvcr.io

export LOCAL_NIM_CACHE=~/.cache/llama-nemotron-topic-guard
mkdir -p "${LOCAL_NIM_CACHE}"
chmod 700 "${LOCAL_NIM_CACHE}"

docker run -d --name llama-nemotron-topic-guard \
  --gpus=all --runtime=nvidia --shm-size=64GB \
  -e NGC_API_KEY \
  -u $(id -u) \
  -v "${LOCAL_NIM_CACHE}:/opt/nim/.cache/" \
  -p 8123:8000 \
  nvcr.io/nim/nvidia/llama-3.1-nemoguard-8b-topic-control:1.10.1
```

**Llama 3.1 8B Instruct NIM** (port 8001):

```bash
docker run -d --name llama-3.1-8b-instruct \
  --gpus=all --runtime=nvidia \
  -e NGC_API_KEY \
  -p 8001:8000 \
  nvcr.io/nim/meta/llama-3.1-8b-instruct:latest
```

Wait until both containers log `Application startup complete`, then set `DEPLOYMENT = 'local'` in the **Choose Deployment Type** cell below and run the remaining cells.

## Remote Deployment

Set your NVIDIA API key before running the config cells:

```bash
export NVIDIA_API_KEY="nvapi-..."
```

You can obtain an API key at [build.nvidia.com](https://build.nvidia.com).

Set `DEPLOYMENT = 'remote'` in the **Choose Deployment Type** cell below and run the remaining cells.

## Choose Deployment Type

Set `DEPLOYMENT` to `'local'` if you completed the **Local Deployment** setup above, or `'remote'` if you are using the NVIDIA-hosted endpoint.

In [1]:
DEPLOYMENT = "remote"
assert DEPLOYMENT in ("local", "remote"), "DEPLOYMENT must be 'local' or 'remote'"

## Import the Necessary Modules

In [2]:
import nest_asyncio

from nemoguardrails import LLMRails, RailsConfig

nest_asyncio.apply()

## Topic Control

The topic control input rail classifies each user message as `on-topic` or `off-topic` based on a set of guidelines you define. Off-topic requests are blocked before reaching the main LLM.

The guidelines are passed to the TopicControl NIM as a system prompt. You can customize them for any domain — the example below configures a customer service assistant for a software company.

### Input rail

In [3]:
# For remote deployment:
# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

YAML_CONFIG = """
models:
  - type: main
    engine: nim
    model: meta/llama-3.1-8b-instruct

  - type: topic_control
    engine: nim
    model: nvidia/llama-3.1-nemoguard-8b-topic-control

rails:
  input:
    flows:
      - topic safety check input $model=topic_control

prompts:
  - task: topic_safety_check_input $model=topic_control
    content: |
      You are a customer service assistant for a software company. Your role is to
      help customers with product questions, technical support, account management,
      billing inquiries, and subscription changes.

      Guidelines:
      - Do not answer questions unrelated to the company or its software products.
      - Do not answer questions about politics, religion, or other sensitive topics.
      - Do not provide personal opinions or advice beyond your customer support role.
      - Do not answer questions asking for personal details about the agent or its creators.
      - Allow general small talk and greetings.
      - For off-topic requests, politely redirect the conversation.
"""

config = RailsConfig.from_content(yaml_content=YAML_CONFIG)

if DEPLOYMENT == "local":
    config.models[0].parameters["base_url"] = "http://localhost:8001/v1"
    config.models[1].parameters["base_url"] = "http://localhost:8123/v1"
    config.models[1].parameters["model_name"] = "nvidia/llama-3.1-nemoguard-8b-topic-control"
elif DEPLOYMENT == "remote":
    config.models[0].api_key_env_var = "NVIDIA_API_KEY"
    config.models[1].api_key_env_var = "NVIDIA_API_KEY"

rails = LLMRails(config)

### Blocking an off-topic request

The guidelines explicitly prohibit political questions. The topic control rail classifies this as off-topic and blocks it before the main LLM generates a response.

In [4]:
response = rails.generate(
    messages=[{"role": "user", "content": "Which party should I vote for in the next election?"}]
)
info = rails.explain()

print("Response")
print("----------------------------------------")
print(response["content"])

print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()

Response
----------------------------------------
I'm sorry, I can't respond to that.


Colang history
----------------------------------------
execute topic_safety_check_input
# The result was {'on_topic': False}
bot refuse to respond
  "I'm sorry, I can't respond to that."
bot stop



LLM calls summary
----------------------------------------
Summary: 1 LLM call(s) took 0.45 seconds and used 239 tokens.

1. Task `topic_safety_check_input $model=topic_control` took 0.45 seconds and used 239 tokens.



### Passing an on-topic request

A subscription question is within the allowed topics, so the topic control rail passes it to the main LLM for a response.

In [5]:
response = rails.generate(
    messages=[{"role": "user", "content": "I'd like to cancel my subscription. Can I do this by phone or on the website?"}]
)
info = rails.explain()

print("Response")
print("----------------------------------------")
print(response["content"])

print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()

Response
----------------------------------------
I'm happy to help you with that. To cancel your subscription, you have a couple of convenient options. You can definitely do it both by phone and on our website, whichever is more comfortable for you.

If you'd like to cancel over the phone, you can simply call our dedicated customer service number, which is available 24/7. The number is 1-800-MY-ACCOUNT (1-800-692-2266). Our friendly and knowledgeable representatives will be happy to assist you with the cancellation process. They'll ask you to provide some basic information to verify your account, and then they'll take care of the rest. The call should only take a few minutes, and you'll be all set.

On the other hand, if you'd prefer to cancel online, you can easily do so through our website. Just log in to your account, navigate to the 'Account Settings' or 'Subscription' section, and look for the 'Cancel Subscription' or 'Cancel Account' button. Follow the prompts, and you'll be gui